# Datathon Passos Mágicos — Limpeza e Qualidade dos Dados

## FIAP Pós-Tech — Fase 5

Este notebook prepara os dados do PEDE de 2022, 2023 e 2024 para a análise exploratória e para o modelo preditivo.

As principais etapas são:

- carregamento e auditoria das três abas;
- seleção e padronização das variáveis;
- correção de tipos e valores inconsistentes;
- criação de uma base única no formato aluno × ano;
- criação de colunas padronizadas para fase, pedra e instituição;
- validação final e exportação do arquivo `dados_pede.csv`.

Os dados ausentes não serão preenchidos indiscriminadamente. Ausências estruturais serão preservadas e tratadas posteriormente conforme a finalidade de cada análise.

In [1]:
from pathlib import Path
from datetime import datetime, date
from IPython.display import display

import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.float_format", "{:.2f}".format)

print("Bibliotecas carregadas com sucesso.")

/home/oai/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-xr7tevyi because there was an issue with the default path (/home/oai/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Bibliotecas carregadas com sucesso.


## 1. Localização e carregamento da planilha

In [2]:
pasta_atual = Path.cwd()

candidatos = sorted(
    pasta_atual.glob("BASE DE DADOS PEDE 2024 - DATATHON*.xlsx")
)

if not candidatos:
    raise FileNotFoundError(
        "A planilha não foi encontrada. Coloque o arquivo Excel na mesma pasta do notebook."
    )

arquivo_padrao = pasta_atual / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

if arquivo_padrao.exists():
    arquivo = arquivo_padrao
else:
    arquivo = candidatos[0]

print(f"Arquivo encontrado: {arquivo.resolve()}")

Arquivo encontrado: /mnt/data/test_final_nb/BASE DE DADOS PEDE 2024 - DATATHON.xlsx


In [3]:
arquivo_excel = pd.ExcelFile(arquivo)

print("Abas encontradas:")
print(arquivo_excel.sheet_names)

abas_esperadas = {"PEDE2022", "PEDE2023", "PEDE2024"}

if not abas_esperadas.issubset(set(arquivo_excel.sheet_names)):
    raise ValueError(
        "A planilha não contém todas as abas esperadas: PEDE2022, PEDE2023 e PEDE2024."
    )

Abas encontradas:
['PEDE2022', 'PEDE2023', 'PEDE2024']


In [4]:
df_2022 = pd.read_excel(arquivo, sheet_name="PEDE2022")
df_2023 = pd.read_excel(arquivo, sheet_name="PEDE2023")
df_2024 = pd.read_excel(arquivo, sheet_name="PEDE2024")

bases = {
    2022: df_2022,
    2023: df_2023,
    2024: df_2024
}

for ano, df in bases.items():
    print(f"{ano}: {df.shape[0]} linhas e {df.shape[1]} colunas")

2022: 860 linhas e 42 colunas
2023: 1014 linhas e 48 colunas
2024: 1156 linhas e 50 colunas


In [5]:
resumo_dimensoes = pd.DataFrame(
    [
        {
            "ano": ano,
            "quantidade_registros": df.shape[0],
            "quantidade_colunas": df.shape[1],
            "celulas_ausentes": int(df.isna().sum().sum()),
            "linhas_duplicadas": int(df.duplicated().sum()),
            "ra_ausente": int(df["RA"].isna().sum()),
            "ra_duplicado_no_ano": int(df["RA"].duplicated().sum())
        }
        for ano, df in bases.items()
    ]
)

resumo_dimensoes

,ano,quantidade_registros,quantidade_colunas,celulas_ausentes,linhas_duplicadas,ra_ausente,ra_duplicado_no_ano
0,2022,860,42,2956,0,0,0
1,2023,1014,48,21157,0,0,0
2,2024,1156,50,21929,0,0,0


### Conclusão da carga inicial

A planilha possui três bases anuais com estruturas diferentes. Por isso, as colunas comuns precisam ser renomeadas e reorganizadas antes da união dos anos.

## 2. Auditoria da estrutura original

In [6]:
for ano, df in bases.items():
    colunas_com_sufixo = [
        coluna
        for coluna in df.columns
        if str(coluna).endswith(".1")
    ]

    print(f"{ano} — possíveis nomes duplicados: {colunas_com_sufixo}")

2022 — possíveis nomes duplicados: []
2023 — possíveis nomes duplicados: ['Destaque IPV.1']
2024 — possíveis nomes duplicados: ['Ativo/ Inativo.1']


In [7]:
resumo_nulos_originais = {}

for ano, df in bases.items():
    tabela = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "quantidade_nulos": df.isna().sum(),
        "percentual_nulos": (df.isna().mean() * 100).round(2)
    }).sort_values("percentual_nulos", ascending=False)

    resumo_nulos_originais[ano] = tabela

    print(f"Maiores percentuais de ausência em {ano}:")
    display(tabela.head(10))

Maiores percentuais de ausência em 2022:


,tipo,quantidade_nulos,percentual_nulos
Inglês,float64,577,67.09
Rec Av4,object,564,65.58
Avaliador4,object,550,63.95
Pedra 20,object,537,62.44
Pedra 21,object,398,46.28
Avaliador3,object,326,37.91
Matem,float64,2,0.23
Portug,float64,2,0.23
Fase,int64,0,0.00
RA,object,0,0.00


Maiores percentuais de ausência em 2023:


,tipo,quantidade_nulos,percentual_nulos
Pedra 23,float64,1014,100.00
Rec Psicologia,float64,1014,100.00
Indicado,float64,1014,100.00
Atingiu PV,float64,1014,100.00
Destaque IEG,float64,1014,100.00
Destaque IDA,float64,1014,100.00
Destaque IPV,float64,1014,100.00
Destaque IPV.1,float64,1014,100.00
Rec Av2,float64,1014,100.00
Rec Av3,float64,1014,100.00


Maiores percentuais de ausência em 2024:


,tipo,quantidade_nulos,percentual_nulos
Cg,float64,1156,100.00
Cf,float64,1156,100.00
Atingiu PV,float64,1156,100.00
Indicado,float64,1156,100.00
Destaque IEG,float64,1156,100.00
Rec Psicologia,float64,1156,100.00
Rec Av1,float64,1156,100.00
Rec Av2,float64,1156,100.00
Destaque IPV,float64,1156,100.00
Destaque IDA,float64,1156,100.00


## 3. Padronização das variáveis anuais

Cada linha da base final representará um aluno em determinado ano. As variáveis recebem os mesmos nomes nos três períodos.

In [8]:
pede_2022 = pd.DataFrame({
    "ra": df_2022["RA"],
    "ano": 2022,
    "fase": df_2022["Fase"],
    "turma": df_2022["Turma"],
    "nome": df_2022["Nome"],
    "idade": df_2022["Idade 22"],
    "genero": df_2022["Gênero"],
    "ano_ingresso": df_2022["Ano ingresso"],
    "instituicao": df_2022["Instituição de ensino"],
    "pedra": df_2022["Pedra 22"],
    "inde": df_2022["INDE 22"],
    "iaa": df_2022["IAA"],
    "ieg": df_2022["IEG"],
    "ips": df_2022["IPS"],
    "ipp": np.nan,
    "ida": df_2022["IDA"],
    "nota_matematica": df_2022["Matem"],
    "nota_portugues": df_2022["Portug"],
    "nota_ingles": df_2022["Inglês"],
    "indicado_bolsa": df_2022["Indicado"],
    "atingiu_pv": df_2022["Atingiu PV"],
    "ipv": df_2022["IPV"],
    "ian": df_2022["IAN"],
    "fase_ideal": df_2022["Fase ideal"],
    "defasagem": df_2022["Defas"]
})

pede_2023 = pd.DataFrame({
    "ra": df_2023["RA"],
    "ano": 2023,
    "fase": df_2023["Fase"],
    "turma": df_2023["Turma"],
    "nome": df_2023["Nome Anonimizado"],
    "idade": df_2023["Idade"],
    "genero": df_2023["Gênero"],
    "ano_ingresso": df_2023["Ano ingresso"],
    "instituicao": df_2023["Instituição de ensino"],
    "pedra": df_2023["Pedra 2023"],
    "inde": df_2023["INDE 2023"],
    "iaa": df_2023["IAA"],
    "ieg": df_2023["IEG"],
    "ips": df_2023["IPS"],
    "ipp": df_2023["IPP"],
    "ida": df_2023["IDA"],
    "nota_matematica": df_2023["Mat"],
    "nota_portugues": df_2023["Por"],
    "nota_ingles": df_2023["Ing"],
    "indicado_bolsa": df_2023["Indicado"],
    "atingiu_pv": df_2023["Atingiu PV"],
    "ipv": df_2023["IPV"],
    "ian": df_2023["IAN"],
    "fase_ideal": df_2023["Fase Ideal"],
    "defasagem": df_2023["Defasagem"]
})

pede_2024 = pd.DataFrame({
    "ra": df_2024["RA"],
    "ano": 2024,
    "fase": df_2024["Fase"],
    "turma": df_2024["Turma"],
    "nome": df_2024["Nome Anonimizado"],
    "idade": df_2024["Idade"],
    "genero": df_2024["Gênero"],
    "ano_ingresso": df_2024["Ano ingresso"],
    "instituicao": df_2024["Instituição de ensino"],
    "pedra": df_2024["Pedra 2024"],
    "inde": df_2024["INDE 2024"],
    "iaa": df_2024["IAA"],
    "ieg": df_2024["IEG"],
    "ips": df_2024["IPS"],
    "ipp": df_2024["IPP"],
    "ida": df_2024["IDA"],
    "nota_matematica": df_2024["Mat"],
    "nota_portugues": df_2024["Por"],
    "nota_ingles": df_2024["Ing"],
    "indicado_bolsa": df_2024["Indicado"],
    "atingiu_pv": df_2024["Atingiu PV"],
    "ipv": df_2024["IPV"],
    "ian": df_2024["IAN"],
    "fase_ideal": df_2024["Fase Ideal"],
    "defasagem": df_2024["Defasagem"]
})

print("Bases anuais padronizadas.")

Bases anuais padronizadas.


In [9]:
df_pede = pd.concat(
    [pede_2022, pede_2023, pede_2024],
    ignore_index=True
)

print(f"Linhas: {df_pede.shape[0]}")
print(f"Colunas iniciais: {df_pede.shape[1]}")

df_pede.head()

Linhas: 3030
Colunas iniciais: 25


,ra,ano,fase,turma,nome,idade,genero,ano_ingresso,instituicao,pedra,inde,iaa,ieg,ips,ipp,ida,nota_matematica,nota_portugues,nota_ingles,indicado_bolsa,atingiu_pv,ipv,ian,fase_ideal,defasagem
0,RA-1,2022,7,A,Aluno-1,19,Menina,2016,Escola Pública,Quartzo,5.78,8.30,4.10,5.60,NaN,4.00,2.70,3.50,6.00,Sim,Não,7.28,5.00,Fase 8 (Universitários),-1
1,RA-2,2022,7,A,Aluno-2,17,Menina,2017,Rede Decisão,Ametista,7.05,8.80,5.20,6.30,NaN,6.80,6.30,4.50,9.70,Não,Não,6.78,10.00,Fase 7 (3º EM),0
2,RA-3,2022,7,A,Aluno-3,17,Menina,2016,Rede Decisão,Ágata,6.59,0.00,7.90,5.60,NaN,5.60,5.80,4.00,6.90,Não,Não,7.56,10.00,Fase 7 (3º EM),0
3,RA-4,2022,7,A,Aluno-4,17,Menino,2017,Rede Decisão,Quartzo,5.95,8.80,4.50,5.60,NaN,5.00,2.80,3.50,8.70,Não,Não,5.28,10.00,Fase 7 (3º EM),0
4,RA-5,2022,7,A,Aluno-5,17,Menina,2016,Rede Decisão,Ametista,7.43,7.90,8.60,5.60,NaN,5.20,7.00,2.90,5.70,Não,Não,7.39,10.00,Fase 7 (3º EM),0


## 4. Limpeza de textos, números e idades

In [10]:
colunas_texto = [
    "ra",
    "fase",
    "turma",
    "nome",
    "genero",
    "instituicao",
    "pedra",
    "indicado_bolsa",
    "atingiu_pv",
    "fase_ideal"
]

for coluna in colunas_texto:
    df_pede[coluna] = (
        df_pede[coluna]
        .astype("string")
        .str.strip()
    )

df_pede["genero"] = df_pede["genero"].replace({
    "Menina": "Feminino",
    "Menino": "Masculino"
})

df_pede["instituicao"] = df_pede["instituicao"].replace({
    "Escola Pública": "Pública"
})

print("Campos de texto padronizados.")

Campos de texto padronizados.


In [11]:
def corrigir_idade(valor):
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, (datetime, date, pd.Timestamp)):
        return valor.day

    return pd.to_numeric(valor, errors="coerce")


df_pede["idade"] = df_pede["idade"].apply(corrigir_idade)

print("Idades corrigidas.")

Idades corrigidas.


In [12]:
colunas_numericas = [
    "idade",
    "ano_ingresso",
    "inde",
    "iaa",
    "ieg",
    "ips",
    "ipp",
    "ida",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "ipv",
    "ian",
    "defasagem"
]

for coluna in colunas_numericas:
    df_pede[coluna] = pd.to_numeric(
        df_pede[coluna],
        errors="coerce"
    )

quantidade_1865 = int(
    (df_pede[colunas_numericas] == 1865).sum().sum()
)

df_pede[colunas_numericas] = (
    df_pede[colunas_numericas]
    .replace(1865, np.nan)
)

print(f"Valores iguais a 1865 encontrados e tratados: {quantidade_1865}")

Valores iguais a 1865 encontrados e tratados: 0


In [13]:
resumo_idade = (
    df_pede
    .groupby("ano")["idade"]
    .agg(
        total_registros="size",
        valores_preenchidos="count",
        valores_nulos=lambda valores: valores.isna().sum(),
        idade_minima="min",
        idade_maxima="max",
        idade_media="mean"
    )
    .round(2)
)

resumo_idade

,total_registros,valores_preenchidos,valores_nulos,idade_minima,idade_maxima,idade_media
ano,,,,,,
2022,860,860,0,7,21,12.14
2023,1014,1014,0,7,26,12.33
2024,1156,1156,0,7,27,12.99


## 5. Padronização de fase, pedra e instituição

In [14]:
def extrair_numero_fase(valor):
    if pd.isna(valor):
        return np.nan

    texto = str(valor).strip().upper()

    if texto.startswith("ALFA"):
        return 0

    numero = re.search(r"\d+", texto)

    if numero:
        return int(numero.group())

    return np.nan


df_pede["fase_num"] = (
    df_pede["fase"]
    .apply(extrair_numero_fase)
    .astype("Int64")
)

df_pede["fase_ideal_num"] = (
    df_pede["fase_ideal"]
    .apply(extrair_numero_fase)
    .astype("Int64")
)

pd.DataFrame({
    "fase_atual": df_pede["fase_num"].value_counts().sort_index(),
    "fase_ideal": df_pede["fase_ideal_num"].value_counts().sort_index()
}).fillna(0).astype(int)

,fase_atual,fase_ideal
0,617,236
1,550,386
2,540,745
3,491,644
4,285,271
5,225,224
6,76,177
7,81,147
8,127,200
9,38,0


In [15]:
df_pede["pedra_padrao"] = (
    df_pede["pedra"]
    .replace({
        "Agata": "Ágata",
        "INCLUIR": pd.NA
    })
    .astype("string")
)

df_pede["instituicao_padrao"] = (
    df_pede["instituicao"]
    .replace({
        "Escola Pública": "Pública",
        "Privada - Programa de apadrinhamento":
            "Privada - Programa de Apadrinhamento"
    })
    .astype("string")
)

print("Pedras padronizadas:")
display(df_pede["pedra_padrao"].value_counts(dropna=False).to_frame("quantidade"))

print("Instituições padronizadas:")
display(df_pede["instituicao_padrao"].value_counts(dropna=False).to_frame("quantidade"))

Pedras padronizadas:


,quantidade
pedra_padrao,
Ametista,1120
Ágata,721
Topázio,688
Quartzo,316
<NA>,185


Instituições padronizadas:


,quantidade
instituicao_padrao,
Pública,2474
Privada - Programa de Apadrinhamento,196
Rede Decisão,106
Privada,104
Privada *Parcerias com Bolsa 100%,101
Privada - Pagamento por *Empresa Parceira,17
Concluiu o 3º EM,14
Bolsista Universitário *Formado (a),13
Escola JP II,2


## 6. Validação e correção dos indicadores

In [16]:
indicadores = [
    "inde",
    "iaa",
    "ieg",
    "ips",
    "ipp",
    "ida",
    "ipv",
    "ian"
]

resumo_indicadores_antes = (
    df_pede
    .groupby("ano")[indicadores]
    .agg(["count", "min", "max", "mean"])
    .round(2)
)

resumo_indicadores_antes

inde                  iaa                   ieg                   ips  \
     count  min  max mean count  min   max mean count  min   max mean count   
ano                                                                           
2022   860 3.03 9.44 7.04   860 0.00 10.00 8.27   860 0.00 10.00 7.89   860   
2023   931 3.75 9.37 7.34   951 0.00 10.00 6.90   938 3.70 10.00 8.70   945   
2024  1054 3.79 9.53 7.40  1054 0.00 10.00 8.54  1156 0.00 10.00 7.37  1054   

                       ipp                   ida                   ipv       \
      min   max mean count  min   max mean count  min   max mean count  min   
ano                                                                           
2022 2.50 10.00 6.90     0  NaN   NaN  NaN   860 0.00  9.90 6.09   860 2.50   
2023 2.52 10.00 5.12   938 3.75  9.79 7.56   937 0.00 10.00 6.66   938 3.32   
2024 2.51 10.00 6.83  1054 2.50 10.00 7.55  1055 0.00 10.00 6.35  1054 2.94   

                  ian                  
       max mean count  min   max mean  
ano                                    
2022 10.00 7.25   860 2.50 10.00 6.42  
2023 10.01 8.03  1014 2.50 10.00 7.24  
2024  9.76 7.35  1156 2.50 10.00 7.68

In [17]:
df_pede["iaa_original"] = df_pede["iaa"]
df_pede["ipv_original"] = df_pede["ipv"]

ajustes_indicadores = {}

for coluna in ["iaa", "ipv"]:
    filtro_pequeno_excesso = (
        (df_pede[coluna] > 10)
        & (df_pede[coluna] <= 10.1)
    )

    ajustes_indicadores[coluna] = int(
        filtro_pequeno_excesso.sum()
    )

    df_pede.loc[
        filtro_pequeno_excesso,
        coluna
    ] = 10

print("Pequenos excessos corrigidos:")
print(ajustes_indicadores)

Pequenos excessos corrigidos:
{'iaa': 122, 'ipv': 26}


In [18]:
problemas_indicadores = []

for coluna in indicadores:
    filtro = (
        df_pede[coluna].notna()
        & ~df_pede[coluna].between(0, 10)
    )

    if filtro.sum() > 0:
        problemas_indicadores.append({
            "coluna": coluna,
            "quantidade": int(filtro.sum()),
            "menor_valor": df_pede.loc[filtro, coluna].min(),
            "maior_valor": df_pede.loc[filtro, coluna].max()
        })

if problemas_indicadores:
    display(pd.DataFrame(problemas_indicadores))
else:
    print("Não existem indicadores fora da escala de 0 a 10.")

Não existem indicadores fora da escala de 0 a 10.


## 7. Verificação da defasagem

In [19]:
df_pede["defasagem_original"] = df_pede["defasagem"]

df_pede["defasagem_calculada"] = (
    df_pede["fase_num"]
    - df_pede["fase_ideal_num"]
).astype("Int64")

filtro_defasagem_inconsistente = (
    df_pede["defasagem"].notna()
    & df_pede["defasagem_calculada"].notna()
    & (
        df_pede["defasagem"]
        != df_pede["defasagem_calculada"]
    )
)

inconsistencias_defasagem = df_pede.loc[
    filtro_defasagem_inconsistente,
    [
        "ra",
        "ano",
        "fase",
        "fase_num",
        "fase_ideal",
        "fase_ideal_num",
        "ian",
        "defasagem",
        "defasagem_calculada"
    ]
].copy()

print(
    f"Inconsistências encontradas antes da correção: "
    f"{len(inconsistencias_defasagem)}"
)

display(inconsistencias_defasagem)

Inconsistências encontradas antes da correção: 2


,ra,ano,fase,fase_num,fase_ideal,fase_ideal_num,ian,defasagem,defasagem_calculada
2444,RA-1516,2024,3A,3,Fase 3 (7° e 8° ano),3,10.00,3,0
2452,RA-1519,2024,3A,3,Fase 3 (7° e 8° ano),3,10.00,3,0


In [20]:
df_pede.loc[
    filtro_defasagem_inconsistente,
    "defasagem"
] = (
    df_pede.loc[
        filtro_defasagem_inconsistente,
        "defasagem_calculada"
    ]
    .astype(float)
)

defasagem_informada = pd.to_numeric(
    df_pede["defasagem"],
    errors="coerce"
).to_numpy(dtype=float)

defasagem_calculada = pd.to_numeric(
    df_pede["defasagem_calculada"],
    errors="coerce"
).to_numpy(dtype=float)

quantidade_restante = int(
    (~np.isclose(
        defasagem_informada,
        defasagem_calculada,
        equal_nan=True
    )).sum()
)

print(
    f"Inconsistências restantes após a correção: "
    f"{quantidade_restante}"
)

Inconsistências restantes após a correção: 0


## 8. Inconsistências longitudinais

Algumas informações que deveriam ser relativamente estáveis mudam entre os anos. Esses registros serão mantidos, mas receberão flags para que possam ser excluídos de análises específicas.

In [21]:
generos_por_aluno = (
    df_pede
    .groupby("ra")["genero"]
    .nunique(dropna=True)
)

alunos_genero_inconsistente = (
    generos_por_aluno[
        generos_por_aluno > 1
    ].index
)

df_pede["genero_inconsistente"] = (
    df_pede["ra"]
    .isin(alunos_genero_inconsistente)
)


ingressos_por_aluno = (
    df_pede
    .groupby("ra")["ano_ingresso"]
    .nunique(dropna=True)
)

alunos_ingresso_inconsistente = (
    ingressos_por_aluno[
        ingressos_por_aluno > 1
    ].index
)

df_pede["ano_ingresso_inconsistente"] = (
    df_pede["ra"]
    .isin(alunos_ingresso_inconsistente)
)


idades_ordenadas = (
    df_pede[
        ["ra", "ano", "idade"]
    ]
    .sort_values(["ra", "ano"])
    .copy()
)

idades_ordenadas["diferenca_ano"] = (
    idades_ordenadas
    .groupby("ra")["ano"]
    .diff()
)

idades_ordenadas["diferenca_idade"] = (
    idades_ordenadas
    .groupby("ra")["idade"]
    .diff()
)

filtro_idade_inconsistente = (
    idades_ordenadas["diferenca_idade"].notna()
    & (
        (idades_ordenadas["diferenca_idade"] < 0)
        |
        (
            idades_ordenadas["diferenca_idade"]
            > idades_ordenadas["diferenca_ano"] + 1
        )
    )
)

alunos_idade_inconsistente = (
    idades_ordenadas.loc[
        filtro_idade_inconsistente,
        "ra"
    ]
    .unique()
)

df_pede["idade_inconsistente"] = (
    df_pede["ra"]
    .isin(alunos_idade_inconsistente)
)

resumo_inconsistencias = pd.DataFrame({
    "tipo": [
        "Gênero",
        "Ano de ingresso",
        "Evolução da idade"
    ],
    "quantidade_alunos": [
        len(alunos_genero_inconsistente),
        len(alunos_ingresso_inconsistente),
        len(alunos_idade_inconsistente)
    ]
})

resumo_inconsistencias

,tipo,quantidade_alunos
0,Gênero,12
1,Ano de ingresso,201
2,Evolução da idade,4


## 9. Valores ausentes após a limpeza

Os valores ausentes são apresentados por ano. Eles não serão preenchidos nesta etapa, pois algumas ausências decorrem da própria estrutura das pesquisas anuais.

In [22]:
nulos_por_ano = pd.concat(
    {
        ano: dados.isna().mean() * 100
        for ano, dados in df_pede.groupby("ano")
    },
    axis=1
).round(2)

nulos_por_ano.columns.name = "ano"

nulos_por_ano.sort_values(
    by=[2022, 2023, 2024],
    ascending=False
)

ano,2022,2023,2024
ipp,100.00,7.50,8.82
nota_ingles,67.09,67.06,59.00
nota_portugues,0.23,7.59,9.17
nota_matematica,0.23,7.59,9.08
indicado_bolsa,0.00,100.00,100.00
atingiu_pv,0.00,100.00,100.00
inde,0.00,8.19,8.82
pedra_padrao,0.00,8.19,8.82
pedra,0.00,8.19,5.54
ida,0.00,7.59,8.74


## 10. Testes finais e exportação

In [23]:
assert df_pede.shape[0] == 3030, (
    "A quantidade de registros foi alterada."
)

assert df_pede.duplicated(
    subset=["ra", "ano"]
).sum() == 0, (
    "Existem alunos duplicados dentro do mesmo ano."
)

assert df_pede["ra"].isna().sum() == 0, (
    "Existem identificadores RA ausentes."
)

assert df_pede["idade"].isna().sum() == 0, (
    "Ainda existem idades ausentes."
)

assert df_pede["idade"].between(5, 30).all(), (
    "Existem idades fora do intervalo esperado."
)

assert df_pede["fase_num"].isna().sum() == 0, (
    "Existem fases que não foram padronizadas."
)

assert df_pede["fase_num"].between(0, 9).all(), (
    "Existem fases atuais inválidas."
)

assert df_pede["fase_ideal_num"].isna().sum() == 0, (
    "Existem fases ideais que não foram padronizadas."
)

assert df_pede["fase_ideal_num"].between(0, 8).all(), (
    "Existem fases ideais inválidas."
)

pedras_validas = {
    "Quartzo",
    "Ágata",
    "Ametista",
    "Topázio"
}

assert set(
    df_pede["pedra_padrao"]
    .dropna()
    .unique()
).issubset(pedras_validas), (
    "Foram encontradas categorias de pedra inválidas."
)

for coluna in indicadores:
    assert (
        df_pede[coluna]
        .dropna()
        .between(0, 10)
        .all()
    ), f"A coluna {coluna} possui valores fora da escala."

assert np.allclose(
    pd.to_numeric(
        df_pede["defasagem"],
        errors="coerce"
    ).to_numpy(dtype=float),
    pd.to_numeric(
        df_pede["defasagem_calculada"],
        errors="coerce"
    ).to_numpy(dtype=float),
    equal_nan=True
), "Ainda existem defasagens inconsistentes."

print("Todos os testes finais foram aprovados.")

Todos os testes finais foram aprovados.


In [24]:
nome_arquivo_saida = "dados_pede.csv"
caminho_saida = Path.cwd() / nome_arquivo_saida

df_pede.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

df_teste = pd.read_csv(caminho_saida)

assert df_teste.shape == df_pede.shape, (
    "O arquivo exportado possui dimensão diferente da base em memória."
)

print("CSV gerado com sucesso.")
print(f"Arquivo: {caminho_saida.resolve()}")
print(f"Linhas: {df_pede.shape[0]}")
print(f"Colunas: {df_pede.shape[1]}")
print(f"Tamanho: {caminho_saida.stat().st_size:,} bytes")

CSV gerado com sucesso.
Arquivo: /mnt/data/test_final_nb/dados_pede.csv
Linhas: 3030
Colunas: 36
Tamanho: 640,832 bytes


## Conclusão

A base final possui uma linha por aluno e ano. As principais variáveis foram padronizadas, as idades foram corrigidas, as fases foram convertidas para uma escala numérica comum, as categorias de pedra foram unificadas e as inconsistências de defasagem foram corrigidas.

O arquivo `dados_pede.csv` está pronto para ser utilizado no notebook de análise exploratória. O tratamento de valores ausentes necessário para Machine Learning será realizado posteriormente dentro do pipeline de modelagem, após a separação entre treino e teste.